In [1]:
import pyGinkgo as pg

### Direct solver bindings

In [ ]:
fn = "m1.mtx"
executor = pg.device("cuda") # alternatives: cpu,hip,omp
mtx = pg.read(path=fn, dtype="double", format="Csr", device=executor)
n_rows = mtx.shape[0]

b = pg.dense(dim=(n_rows, 1), device=executor, dtype="double", fill=1.0)
x = pg.dense(dim=(n_rows, 1), device=executor, dtype="double", fill=0.0)

# Create ILU preconditioner
preconditioner = pg.preconditioner.Ilu(executor, mtx)

# Setup GMRES solver
solver = pg.solver.gmres(
    executor,
    mtx,
    preconditioner=preconditioner,
    max_iters=1000,
    krylov_dim=30,
    reduction_factor=1e-06,
)

# Apply
logger, result = solver.apply(b, x)

In [ ]:
result_cpu = result.copy_to_host()

In [4]:
for i in range(10):
    print(result_cpu.at(i, 0), end=", ")

0.6044527378667173, 0.8250277658122595, 0.9154476498969804, 0.9545267011434276, 0.9719330029976926, 0.9798380401250447, 0.9834770837848464, 0.9851691223635274, 0.9859619246205109, 0.9863356545758512, 

### Config solver

In [5]:
args = {
    "type": "solver::Gmres",
    "krylov_dim": 30,
    "preconditioner": {
        "type": "preconditioner::Jacobi",
        "max_block_size": 1
    },
    "criteria": [
        {"type": "Iteration", "max_iters": 1000},
        {
          "type": "ResidualNorm", 
          "reduction_factor": 1e-6, 
          "baseline": "rhs_norm"
        }
    ],
}

In [6]:
b = pg.dense(dim=(n_rows, 1), device=executor, dtype="double", fill=1.0)
x = pg.dense(dim=(n_rows, 1), device=executor, dtype="double", fill=0.0)

solver = pg.generate_solver(mtx, args)

logger, result = solver.apply(b, x)

In [ ]:
result_cpu_2 = result.copy_to_host()

### Checking difference between two solvers

In [8]:
result_cpu_2.add_scaled(
    pg.dense(dim=(1, 1), device=executor, dtype="double", fill=-1),
    result_cpu
)

In [9]:
for i in range(10):
    print(result_cpu_2.at(i, 0), end=", ")

2.36289754429464e-09, -5.441616490742263e-09, -2.8017624886800263e-09, -1.7956048647960188e-08, -4.7882990905634415e-09, -2.7630594034988576e-08, -5.419530824113394e-09, -3.681703142355275e-08, -1.0450708076703563e-08, -5.73116250013328e-08, 